# Lesson 37: CrewAI I — agents, tasks, crews, tools and collaboration 👩‍🚀👨‍🔧
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Explain CrewAI's building blocks: **Agent**, **Task**, **Crew**, **Process**
2. Run a **sequential** crew where agents pass work to each other
3. Give agents **built-in tools** and write your own **custom tools** (two ways)
4. Make agents **collaborate**: delegation and the **hierarchical** process with a manager

**⏱️ Timing:** first crew 20' | tools 20' | collaboration 20'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**. Multi-agent crews make many LLM calls, so this lesson uses `qwen2.5:7b` (about 5 GB to download). For a faster but less reliable run, change it to `qwen2.5:3b`.

## 🎬 An analogy: a film crew
A film isn't made by one person. There's a **director**, a **writer**, a **camera operator**, an **editor** — each with a role, a goal and skills, working on tasks in order.

**CrewAI** (an open-source Python framework) builds AI teams the same way:
| Building block | What it is | Film-crew analogy |
|---|---|---|
| **Agent** | An LLM with a `role`, `goal` and `backstory` (+ tools) | A crew member |
| **Task** | A `description` and an `expected_output`, assigned to an agent | A job on the call sheet |
| **Crew** | Agents + tasks + a process | The whole production |
| **Process** | `sequential` (one after another) or `hierarchical` (a manager assigns work) | The schedule |
| **Tool** | A function an agent can use | Equipment |

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:7b"     # smarter at using tools. "qwen2.5:3b" downloads faster but makes more mistakes
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!pip -q install crewai
import os, warnings; warnings.filterwarnings("ignore")
os.environ["CREWAI_TRACING_ENABLED"] = "false"     # don't send traces anywhere / don't ask about it
os.environ["OTEL_SDK_DISABLED"] = "true"
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model=f"ollama/{MODEL}", base_url="http://localhost:11434", temperature=0.2)
print("✅ CrewAI ready, using", MODEL)

## Part 1: Your first crew — a science-fair team 🔬
Three agents, three tasks, run **in order**. Each task automatically receives the previous task's output as context.

In [ ]:
idea_agent = Agent(
    role="Science Fair Idea Generator",
    goal="Suggest exciting but safe science-fair projects for Year 9 students",
    backstory="You are a creative science teacher who loves cheap, safe experiments with household items.",
    llm=llm, verbose=True)

planner = Agent(
    role="Experiment Planner",
    goal="Turn a project idea into a clear, fair experiment",
    backstory="You are a careful scientist who insists on one variable changed at a time and repeated trials.",
    llm=llm, verbose=True)

safety = Agent(
    role="Safety Officer",
    goal="Make sure every experiment is safe for 14-year-olds",
    backstory="You are the school lab technician. You spot hazards and add sensible precautions.",
    llm=llm, verbose=True)

t1 = Task(description="Suggest ONE science-fair project about {topic}. Give a title and a one-sentence question it answers.",
          expected_output="A title and a research question", agent=idea_agent)
t2 = Task(description="Write a plan for the project: hypothesis, independent/dependent/controlled variables, method in 5 steps.",
          expected_output="A short experiment plan with variables and 5 method steps", agent=planner)
t3 = Task(description="Review the plan for safety. List up to 3 hazards with precautions, then give a final SAFE / NEEDS CHANGES verdict.",
          expected_output="A hazard table and a verdict", agent=safety)

crew = Crew(agents=[idea_agent, planner, safety], tasks=[t1, t2, t3], process=Process.sequential)
result = crew.kickoff(inputs={"topic": "plants and light"})

In [ ]:
print("🏁 FINAL OUTPUT:\n", result.raw)
print("\n— Each task's output is also saved —")
for t in result.tasks_output:
    print(f"\n[{t.agent}]\n{t.raw[:300]}…")

### 🧩 Exercise 1
1. Run it again with `{"topic": "sound"}`. Notice `{topic}` in the task description is filled from `inputs`.
2. Change the safety officer's **backstory** to be extremely strict (or very relaxed). How does the verdict change? Backstory works like a **role prompt** (Lesson 28).
3. Set `verbose=False` on all agents to hide the step-by-step log.

## Part 2: Tools 🛠️
### Built-in tools
The `crewai-tools` package has many ready-made tools: reading files and folders, searching websites, scraping pages, querying databases… (some need API keys, e.g. web search). Let's give an agent file-reading tools.

In [ ]:
!pip -q install crewai-tools
os.makedirs("lab_reports", exist_ok=True)
open("lab_reports/group_a.txt", "w").write("Group A: plants in full sunlight grew 12 cm in 2 weeks. Plants in shade grew 4 cm. We used 3 plants each.")
open("lab_reports/group_b.txt", "w").write("Group B: we watered plants with salt water and fresh water. Fresh water plants grew 9 cm, salt water 1 cm. One plant per group.")

from crewai_tools import DirectoryReadTool, FileReadTool
marker = Agent(role="Science Report Marker", goal="Give fair feedback on lab reports",
               backstory="You check whether experiments were fair: enough repeats, one variable changed, clear results.",
               tools=[DirectoryReadTool(directory="lab_reports"), FileReadTool()], llm=llm, verbose=True)
mark = Task(description="List the files in the lab_reports folder, read each report, and for each group give: "
                        "what they tested, one strength, one improvement.",
            expected_output="Feedback for each group", agent=marker)
print(Crew(agents=[marker], tasks=[mark]).kickoff().raw)

### Custom tools — way 1: the `@tool` decorator
The **docstring** is the tool's description: the agent reads it to decide when to use the tool. Write it carefully!

In [ ]:
from crewai.tools import tool

@tool("Unit converter")
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert between cm, m, km, g and kg. Example: value=12, from_unit='cm', to_unit='m'."""
    factors = {"cm": 0.01, "m": 1, "km": 1000, "g": 0.001, "kg": 1}
    if from_unit not in factors or to_unit not in factors:
        return f"Unknown unit. Use one of {list(factors)}"
    return f"{value} {from_unit} = {value * factors[from_unit] / factors[to_unit]} {to_unit}"

print(convert_units.run(value=12, from_unit="cm", to_unit="m"))     # test the tool by itself first!

### Custom tools — way 2: subclass `BaseTool`
More control: a typed input schema (with Pydantic), validation and state.

In [ ]:
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
import statistics, json

class ResultsInput(BaseModel):
    measurements: list[float] = Field(description="A list of measurements from repeated trials")

class TrialStatsTool(BaseTool):
    name: str = "Trial statistics"
    description: str = "Calculate the mean, range and spread of repeated experiment measurements."
    args_schema: type[BaseModel] = ResultsInput

    def _run(self, measurements: list[float]) -> str:
        if len(measurements) < 2:
            return "Need at least 2 measurements — repeat the experiment!"
        return json.dumps({"mean": round(statistics.mean(measurements), 2),
                           "range": round(max(measurements) - min(measurements), 2),
                           "std_dev": round(statistics.stdev(measurements), 2)})

stats_tool = TrialStatsTool()
print(stats_tool.run(measurements=[12.1, 11.4, 12.9, 12.3]))

In [ ]:
analyst = Agent(role="Data Analyst", goal="Analyse experiment results accurately using tools, never by mental maths",
                backstory="You are a careful data scientist who always uses tools for calculations.",
                tools=[stats_tool, convert_units], llm=llm, verbose=True)
analyse = Task(description="Plant heights in cm after 2 weeks: sunlight [12.1, 11.4, 12.9, 12.3], shade [4.2, 3.8, 4.9, 4.1]. "
                           "Use the statistics tool for each group, convert the mean heights to metres with the unit converter, "
                           "and say whether the difference is bigger than the spread.",
               expected_output="Stats for each group, means in metres, and a one-sentence conclusion", agent=analyst)
print(Crew(agents=[analyst], tasks=[analyse]).kickoff().raw)

### 🧩 Exercise 2
1. Write a custom tool `periodic_table(symbol)` that returns an element's name and atomic number from a small dictionary. Give it to an agent and ask about "Na, Fe and O".
2. Make a tool's docstring **vague** ("does stuff"). Does the agent still use it correctly? Why do descriptions matter so much?

## Part 3: Collaboration 🤝
### Delegation
With `allow_delegation=True`, an agent can **ask a co-worker** for help or hand over part of its task.

In [ ]:
editor = Agent(role="Newsletter Editor", goal="Publish an accurate, fun school newsletter paragraph",
               backstory="You write well but you are not a scientist, so you ask the science expert to check facts.",
               allow_delegation=True, llm=llm, verbose=True)
expert = Agent(role="Science Expert", goal="Check scientific facts and correct mistakes",
               backstory="You are a physics teacher who explains things simply.", allow_delegation=False, llm=llm, verbose=True)
newsletter = Task(description="Write a 4-sentence newsletter paragraph about why we see lightning before we hear thunder. "
                              "Ask the Science Expert to check the science before you finish.",
                  expected_output="A 4-sentence, fact-checked paragraph", agent=editor)
print(Crew(agents=[editor, expert], tasks=[newsletter]).kickoff().raw)

### Hierarchical process — a manager runs the team
Instead of fixed task owners, a **manager agent** (created automatically from `manager_llm`) decides who does what, checks their work and combines results.

In [ ]:
researcher = Agent(role="Researcher", goal="Find key facts", backstory="You collect clear, accurate facts.", llm=llm)
writer = Agent(role="Writer", goal="Write engaging text for teenagers", backstory="You write short, lively content.", llm=llm)
quizzer = Agent(role="Quiz Maker", goal="Write fair multiple-choice questions", backstory="You are a teacher who writes good quizzes.", llm=llm)

lesson_pack = Task(description="Create a mini lesson pack about volcanoes: 3 key facts, a 100-word explanation for Year 8, "
                               "and 2 multiple-choice questions with answers.",
                   expected_output="Facts, explanation and 2 quiz questions")   # no agent assigned — the manager decides!
manager_crew = Crew(agents=[researcher, writer, quizzer], tasks=[lesson_pack],
                    process=Process.hierarchical, manager_llm=llm, verbose=True)
print(manager_crew.kickoff().raw)

### 🧩 Exercise 3
1. Compare **sequential** and **hierarchical**: which gave more control? Which was more flexible? Which made more LLM calls?
2. When could delegation go wrong? (Agents passing work back and forth, costs growing, nobody responsible.)

### 🚀 Challenge
Build a 3-agent **"study guide crew"** for a topic you're studying now: a summariser, a flashcard writer (with a custom tool that formats cards), and a quiz master. Use `{topic}` inputs so classmates can reuse it.

## 📌 Summary
- **Agent** (role, goal, backstory, tools) + **Task** (description, expected output) + **Crew** (process)
- **Sequential** crews pass outputs along; **hierarchical** crews have a manager
- Tools: built-in (`crewai-tools`) or custom (`@tool` or `BaseTool`) — good **descriptions** are essential
- **Delegation** lets agents ask each other for help